# 2 – Frontend-Plugin mit neuem Menüpunkt

Dieses Notebook erstellt ein Frontend-Plugin namens `team-tools`. Es stellt eine eigene Seite unter `/team-tools` bereit. Im neuen Backstage-Frontend wird der linke Navigationseintrag automatisch aus Titel und Icon der Page Extension erzeugt.


> **Voraussetzung:** Die Backstage-Installation liegt unter `~/mybackstage`.
>
> Shell-Zellen werden über Python mit `subprocess` ausgeführt. Vor Änderungen wird jeweils eine Sicherung angelegt.

In [ ]:
from pathlib import Path
ROOT = Path.home() / "mybackstage"
assert ROOT.exists(), f"{ROOT} wurde nicht gefunden"
PLUGIN = ROOT / "plugins/team-tools"
print("Backstage:", ROOT)

## Plugin-Grundgerüst erzeugen

`yarn new` ist interaktiv. Wähle:

- **plugin**
- Plugin ID: **team-tools**
- Owner: beispielsweise **user:default/guest**

Führe die nächste Zelle aus und beantworte die Prompts im Terminal des Notebook-Kernels.

In [ ]:
import subprocess
if not PLUGIN.exists():
    subprocess.run(["yarn", "new"], cwd=ROOT, check=True)
else:
    print("Plugin-Verzeichnis existiert bereits:", PLUGIN)

## Abhängigkeiten für das neue Frontend-System installieren

In [ ]:
import subprocess
subprocess.run(
    ["yarn", "--cwd", "plugins/team-tools", "add",
     "@backstage/frontend-plugin-api", "@backstage/core-components",
     "@remixicon/react"],
    cwd=ROOT,
    check=True,
)

## Plugin-Dateien schreiben

In [ ]:
from pathlib import Path
import shutil, datetime

src = PLUGIN / "src"
(src / "components/TeamToolsPage").mkdir(parents=True, exist_ok=True)

files = {
    src / "routes.ts": """import { createRouteRef } from '@backstage/frontend-plugin-api';

export const rootRouteRef = createRouteRef();
""",
    src / "plugin.tsx": """import {
  createFrontendPlugin,
  PageBlueprint,
} from '@backstage/frontend-plugin-api';
import { RiToolsLine } from '@remixicon/react';
import { rootRouteRef } from './routes';

const teamToolsPage = PageBlueprint.make({
  params: {
    routeRef: rootRouteRef,
    path: '/team-tools',
    title: 'Team Tools',
    icon: <RiToolsLine />,
    loader: () =>
      import('./components/TeamToolsPage').then(m => <m.TeamToolsPage />),
  },
});

export const teamToolsPlugin = createFrontendPlugin({
  pluginId: 'team-tools',
  extensions: [teamToolsPage],
  routes: {
    root: rootRouteRef,
  },
});
""",
    src / "index.ts": """export { teamToolsPlugin as default } from './plugin';
export { rootRouteRef } from './routes';
""",
    src / "components/TeamToolsPage/TeamToolsPage.tsx": """import {
  Content,
  Header,
  InfoCard,
  Page,
} from '@backstage/core-components';

export const TeamToolsPage = () => (
  <Page themeId="tool">
    <Header
      title="Team Tools"
      subtitle="Werkzeuge und Links für das Entwicklungsteam"
    />
    <Content>
      <InfoCard title="Plugin erfolgreich geladen">
        Diese Seite wird durch das lokale Frontend-Plugin team-tools bereitgestellt.
      </InfoCard>
    </Content>
  </Page>
);
""",
    src / "components/TeamToolsPage/index.ts": """export { TeamToolsPage } from './TeamToolsPage';
""",
}

for path, content in files.items():
    if path.exists():
        backup = path.with_suffix(path.suffix + f".bak-{datetime.datetime.now():%Y%m%d-%H%M%S}")
        shutil.copy2(path, backup)
    path.write_text(content)
    print("geschrieben:", path.relative_to(ROOT))

## Feature Discovery prüfen

Mit `app.packages: all` erkennt Backstage lokale Plugin-Pakete automatisch. Es ist normalerweise keine Änderung an `packages/app/src/App.tsx` erforderlich.

In [ ]:
config = (ROOT / "app-config.yaml").read_text()
print("app.packages: all vorhanden:", "packages: all" in config)

## TypeScript und Build prüfen

In [ ]:
import subprocess
subprocess.run(["yarn", "tsc"], cwd=ROOT, check=True)

## Start und Ergebnis

```bash
cd ~/mybackstage
yarn start
```

Erwartetes Ergebnis:

- links erscheint **Team Tools**,
- der Menüpunkt öffnet `/team-tools`,
- die Seite zeigt eine einfache `InfoCard`.